# Demonstration notebook for LCO via P-graph

This notebook demonstrates the use of the P-graph library for Life Cycle Optimization (LCO). The notebook can be tested online using the Colab environment of google.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jppimentell/P-graph_LCO/blob/main/LCO_P-graph.ipynb)

## Introduction
The P-graph (Process Graph) methodology is a systematic framework for designing and optimizing process systems, enabling the synthesis of efficient solutions that minimize resource consumption and operational costs over a product's or process's lifetime.

## Requirements

- Python 3.x
- Python P-graph library (ProcessGraph. https://github.com/tsyet12/Pgraph) 
- Additional dependencies (NumPy, Pandas, Matplotlib for visualization)



### Import of libraries

As a first step the necessary libraries are downloaded.
Necessary libraries: Pandas, networkx, pyplot and ProcessGrapph.
The ProcessGraph library can be downloaded and installed by following the instructions from the [Github page of the project](https://github.com/tsyet12/Pgraph)

In [ ]:
!pip install git+https://github.com/tsyet12/Pgraph --quiet

In [ ]:
#import all libraries
from Pgraph.Pgraph import Pgraph #This is our Pgraph library
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

Clone repository to get sample files

In [ ]:
# Clone the entire repo to get all data files
!git clone https://github.com/jppimentell/P-graph_LCO /content/P-graph_LCO

# Change to that directory so the notebook can find the files
import os
os.chdir('/content/P-graph_LCO')

### Import of Excel file

The excel file describes Matrices **A**, **B**, **Q**, and the weights of the solution in the optimization for the LCA / LCO model
$$ 
\begin{align}
\min & \sum_i ^K \omega_i h_i\\
\\
&\mathbf{A} \mathbf{s}=\mathbf{f}\\
&\mathbf{B} \mathbf{s}= \mathbf{g}\\
&\mathbf{Q} \mathbf{s} =\mathbf{h}
\end{align}
$$
 where $\mathbf{f}$ is the functional unit vector, $\mathbf{g}$ is the inventory vector, $\mathbf{A}$ is the process matrix, $\mathbf{B}$ is the environmental matrix, and $\mathbf{s}$ is the capacity factor (scaling) vector. Moreover, $\mathbf{Q}$ is the characterization matrix, $\mathbf{h}$ is the midpoint environmental impact vector, $h_i$ is the i-th impact assessed in the LCA ($i=1,...,K$), and $\omega_i$ is its weight.

In [ ]:
file_problem="Data_problem_ammonia"
file_problem="Data_problem_lightbulbs_V2"
file_problem="Data_problem_heat"

data_problem=pd.read_excel(f"./{file_problem}.xlsx",sheet_name="A",index_col=0)
data_problem.head()

### Problem specifications

Specification of colors, units of impacts, and maximum number of solutions to generate

In [ ]:
color_inventories=-65536
impact_units="m2/kg"
max_sols=10

## Transformation into P-graph and solution of LCO 

The subsequent code will transform the information of the matrices into P-graph and will utilize algorithm ABB to find the n-best solutions up to a maximum of solutions specified in the variable `max_sols`

In [ ]:
color_nodes={}
color_edges={}
measurement_units=dict()
A_mat=data_problem.iloc[:-4,:-4]
B_mat=pd.read_excel(f"./{file_problem}.xlsx",sheet_name="B",index_col=0).fillna(0)
Q_mat=pd.read_excel(f"./{file_problem}.xlsx",sheet_name="Q",index_col=0).fillna(0)
weights_obj=pd.read_excel(f"./{file_problem}.xlsx",sheet_name="Weights",index_col=0).fillna(0)["Impact"]
max_cap=1e9
max_flow=1e9
processes=list(A_mat.columns)
flows=list(A_mat.index)
inventories=list(B_mat.index)
impacts=list(Q_mat.index)

G=nx.DiGraph()
G.clear()
ME=[]

for process in processes:
    G.add_node(f"O_{process}",names=f"{process}",
               fix_cost=float(data_problem[process]["FixCost"]),
               proportional_cost=float(data_problem[process]["PropCost"]),
               capacity_lower_bound=float(data_problem[process]["minCap"]),
               capacity_upper_bound=float(data_problem[process]["maxCap"]))
    
#Insert flow nodes
for flow in flows:
    if flow in inventories:
        price_flow=-sum([weights_obj[k]*Q_mat[flow][k] for k in impacts]) if sum([weights_obj[k]*Q_mat[flow][k] for k in impacts]) !=1 else -1.0001
    else:
        price_flow=0
    G.add_node(f"M_{flow}",
               names=f"{flow}",
               price=price_flow,                   
               type=data_problem["Type"][flow],
               flow_rate_lower_bound=data_problem["reqFlow"][flow],
               flow_rate_upper_bound=data_problem["maxFlow"][flow])
    
for process in processes:
    for flow in flows:
        if not pd.isna(A_mat[process][flow]):
            weight_m=float(A_mat[process][flow])
            if weight_m>0:
                G.add_edge(f"O_{process}",f"M_{flow}",weight=weight_m)
            elif weight_m<0:
                G.add_edge(f"M_{flow}",f"O_{process}",weight=-weight_m)

#Impact flows
for inv in inventories:
    if inv not in flows:
        G.add_node(f"M_{inv}",
                   names=f"{inv}",
                   type="product",
                   flow_rate_lower_bound=0,
                   price=-sum([weights_obj[k]*Q_mat[inv][k] for k in impacts]) if sum([weights_obj[k]*Q_mat[inv][k] for k in impacts]) !=1 else -1.0001,
                   flow_rate_upper_bound=100000000)
        color_nodes[f"M_{inv}"]=color_inventories
        measurement_units[f"M_{inv}"]="kilogram (kg)"
    
for process in processes:
    for inv in inventories:
        if inv not in flows:
            if not pd.isna(B_mat[process][inv]):
                weight_m=float(B_mat[process][inv])
                if weight_m>0:
                    G.add_edge(f"O_{process}",f"M_{inv}",weight=weight_m)
                    color_edges[f"O_{process}",f"M_{inv}"]=color_inventories
                elif weight_m<0:
                    G.add_edge(f"M_{inv}",f"O_{process}",weight=-weight_m)
                    color_edges[f"M_{inv}",f"O_{process}"]=color_inventories
                

ME= []
P=Pgraph(problem_network=G, mutual_exclusion=ME, solver="INSIDEOUT",max_sol=max_sols)
#P.to_studio("./",file_name=f"{file_problem.split('_')[-1]}.pgsx",verbose=False,payout_period=1,money_units=impact_units,
#           color_nodes=color_nodes,color_edges=color_edges, measurement_units=measurement_units)
# Run the solver
P.run()


### Printing objectve value of solutions and extracting capacities of units

he subsequent cell collects the value of the objective function and the capacity factors of the processes for each solution. The processes' capacities in each solution are shown in a matrix form, where the columns represent the distinct solutions.

In [ ]:
capacities=pd.DataFrame(index=processes,columns=list(range(1,len(P.goolist)+1)),dtype=float).fillna(0)
for solution in range(len(P.goplist)):
    for process in P.goplist[solution]:
        capacities.loc[process[1][2:],solution+1]= float(process[0])
    print(f"Solution {solution+1} Total impact: {P.goolist[solution]} ")
display(capacities)